# Per-gene TWAS weights against a pre-built QtlDataset

## Description

For a single study's `QtlDataset` RDS, run `pecotmr::twasWeightsPipeline(methods = "default", ...)` per gene. Gene-level parallelization is via SoS task fan-out: each task loads the same RDS and computes TWAS weights for one gene. The default method preset gives the standard univariate methods (SuSiE / lasso / elastic-net / etc. as defined by `pecotmr::.twasMethodLookup("default")`).

Optionally pass a pre-fit `--fine-mapping-result` RDS; SuSiE-family TWAS methods will reuse those fits instead of refitting.

## Inputs

- `--qtl-dataset` &mdash; path to the QtlDataset RDS produced by `qtl_dataset.ipynb`.
- `--genes` &mdash; explicit gene-ID list.
- `--cis-window` &mdash; bp window. Default 1,000,000.
- `--fine-mapping-result` &mdash; optional pre-fit FineMappingResult RDS (e.g. the matching output of `fine_mapping.ipynb`); SuSiE-family TWAS methods will reuse the fits.

## Output

- `{cwd}/{study}.{gene}.twas_weights.rds` per gene.


## Example

```bash
sos run pipeline/twas_weights.ipynb twas_weights \
    --cwd output \
    --study ROSMAP_DLPFC \
    --qtl-dataset output/ROSMAP_DLPFC.qtl_dataset.rds \
    --genes ENSG00000060237 ENSG00000234593 \
    --cis-window 1000000
```


In [ ]:
[global]
parameter: cwd = path('output')
parameter: study = str
parameter: qtl_dataset = path
parameter: genes = []
parameter: cis_window = 1000000
parameter: fine_mapping_result = path('.')
parameter: container = ''
parameter: job_size = 1
parameter: walltime = '1h'
parameter: mem = '16G'
parameter: numThreads = 1


In [ ]:
[twas_weights]
input: qtl_dataset, for_each = 'genes'
output: f"{cwd}/{study}.{_genes}.twas_weights.rds"
task: trunk_workers = 1, trunk_size = job_size, walltime = walltime, mem = mem, cores = numThreads, tags = f"{step_name}_{_output:bn}"
bash: expand = '${ }', stderr = f"{_output}.stderr", stdout = f"{_output}.stdout", container = container
    Rscript ${path("code/script/pecotmr_integration/twas_weights.R", absolute=True)} \
        --qtl-dataset ${_input} \
        --gene-id ${_genes} \
        --cis-window ${cis_window} \
        --fine-mapping-result ${fine_mapping_result if fine_mapping_result.is_file() else '""'} \
        --output ${_output}
